# Factor Neutralization

This notebook implements a full factor neutralization workflow for a multi-factor quantitative strategy, including:
- Outlier treatment (MAD-based clipping)
- Standardization (Z-Score)
- Industry neutralization (Shenwan Level-1 industry classification)
- Market-cap neutralization
- Joint industry + market-cap neutralization

By using OLS regression, we remove industry and size effects from raw factors to obtain cleaner alpha signals.

In [ ]:
from atrader import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import DataFrame
from MAD_factor import extreme_MAD
import statsmodels.api as sm

## Get Shenwan Level-1 Industry Mapping

Read records from sw_industry.csv where the industry short code ends with '1' (Shenwan Level-1 industries), and return a dictionary mapping Chinese industry names to short codes.

In [ ]:
def get_industry_codes_ending_with_1():
    """
    Get industry codes from sw_industry.csv that end with '1'.
    
    Returns:
        dict: Dictionary with industry Chinese name (行业中文名) as key 
              and industry short code (行业简称) as value
    """
    import os
    
    current_dir = os.path.dirname(os.path.abspath('__file__'))
    csv_path = os.path.join(current_dir, '../../data/sw_industry.csv')
    
    df = pd.read_csv(csv_path)
    
    filtered_df = df[df['行业简称'].str.endswith('1')]
    
    result_dict = dict(zip(filtered_df['行业中文名'], filtered_df['行业简称']))
    
    return result_dict

## Build Industry Exposure Matrix

Construct an industry dummy matrix (0/1): for each stock code, determine its Shenwan Level-1 industry membership.

In [ ]:
def industry_exposure(target_idx: list):
    shenwan_industry = get_industry_codes_ending_with_1()
    df = pd.DataFrame(index=[x.lower() for x in target_idx], columns=shenwan_industry.keys())
    for industry in df.columns:
        temp = list(set(df.index).intersection(set(get_code_list(shenwan_industry[industry]).code.tolist())))
        df.loc[temp, industry] = 1
    return df.fillna(0)

## Define Neutralization Function

Use OLS regression to neutralize each factor column against industry dummies and/or log market cap. The regression residuals are treated as neutralized factor values. Supported modes:
- Industry-only neutralization
- Market-cap-only neutralization
- Industry + market-cap neutralization

In [ ]:
def neutralization(factor: DataFrame, mkv: DataFrame, industry=True):
    Y = factor.fillna(0)
    Y.rename(index = str.lower, inplace=True)
    df = pd.DataFrame(index=Y.index, columns=Y.columns)
    lnmkv = None
    if (type(mkv) == DataFrame or type(mkv) == pd.Series):
        mkv.rename(index=str.lower, inplace=True)
        lnmkv = mkv.iloc[:, 0].apply(lambda x: np.log(x))
        lnmkv = lnmkv.fillna(0)
    for i in range(Y.columns.size):
        if (lnmkv is not None):
            if industry:
                dummy_industry = industry_exposure(Y.index.tolist())
                X = pd.concat([dummy_industry, lnmkv], axis=1, sort=False)
            else:
                X = lnmkv
        elif industry:
            dummy_industry = industry_exposure(factor.index.tolist())
            X = dummy_industry
            
        result = sm.OLS(Y.iloc[:, i], X).fit()
        df.iloc[:, i] = result.resid.tolist()
    return df

## Define Standardization Function

Apply Z-Score standardization column-wise: subtract the mean and divide by the standard deviation.

In [ ]:
def standardization(df: DataFrame):
    return (df - df.mean()) / df.std()

## Load Stock Universe and Market Cap Data

Set the target date, retrieve HS300 constituent codes, and fetch market-cap factor data.

In [ ]:
date = '2025-07-01'
code_list = list(get_code_list('hs300', date=date).code)
print('code size: {}'.format(len(code_list)))

mkv = get_factor_by_day(factor_list=['market_cap_3'], target_list=code_list, date=date)
mkv.rename(columns={'market_cap_3': 'mkv'}, inplace=True)
mkv.set_index('code', inplace=True)
print(mkv)

## Load Factor Data

Fetch fundamental factors for HS300 constituents: PB, PE, PS, and ROE.

In [ ]:
factor = get_factor_by_day(
    factor_list=['pb_ratio_ttm', 'pe_ratio_ttm', 'ps_ratio_ttm', 'du_return_on_equity_ttm'],
    target_list=code_list,
    date=date
)
factor.rename(columns={
    'pb_ratio_ttm': 'pb',
    'pe_ratio_ttm': 'pe',
    'ps_ratio_ttm': 'ps',
    'du_return_on_equity_ttm': 'roe'
}, inplace=True)
factor.set_index('code', inplace=True)
print(factor)

## Compute Industry Exposure Matrix

Generate the Shenwan Level-1 industry dummy matrix using stock codes from the factor DataFrame.

In [ ]:
ind = industry_exposure(factor.index.tolist())
print(ind)

## Apply Outlier Treatment and Standardization

Use MAD (Median Absolute Deviation) to handle outliers, then apply Z-Score standardization.

In [ ]:
factor_S = standardization(extreme_MAD(factor))

## Run Neutralization Variants (Industry, Market Cap, Both)

Apply three neutralization variants to standardized factors:
1. factor_ID: industry-only neutralization (mkv=0)
2. factor_mkv: market-cap-only neutralization (industry=False)
3. factor_IDmkv: joint industry + market-cap neutralization

In [ ]:
factor_ID = neutralization(factor_S, 0)
factor_mkv = neutralization(factor_S, mkv, industry=False)
factor_IDmkv = neutralization(factor_S, mkv, industry=True)

## Visualize Neutralization Effects

Plot KDE curves of the first factor column across all variants to compare distribution changes after each neutralization method.

In [ ]:
fig = plt.figure(figsize=(14, 8))
factor_S.iloc[:, 0].plot(kind='density', label='factor_S')
factor_ID.iloc[:, 0].plot(kind='density', label='factor_ID')
factor_mkv.iloc[:, 0].plot(kind='density', label='factor_mkv')
factor_IDmkv.iloc[:, 0].plot(kind='density', label='factor_IDmkv')
plt.legend()
plt.show()